# Assay Ground Truth: GEPA-Optimised Stratification with Fixed Validation

This notebook creates stratified data splits optimised for GEPA prompt optimisation experiments.

## Design Overview

**Key Change from Previous Version (DD-2025-003):**
- Previous: Large validation (90%), small training (10%) - inverted from GEPA's requirements
- Current: Fixed small validation (~35 samples), variable training sizes

**GEPA Requirements:**
- Large training set: More examples = more diverse prompt candidates GEPA can synthesise
- Small validation set: Only for Pareto score tracking; large validation = fewer candidates explored

**Stratification Metric:** Total Assay Data Points (TADP)
- Counts extractable assay values from `isolates_with_linking` and `no_isolates_only_assayinformation`
- Does NOT count `isolate_without_linking` (contains only IDs, no assay data)

**Split Structure:** Fixed Validation + Nested Training
- Fixed validation set: ~35 samples (stratified, representative)
- Nested training: 10% ⊂ 20% ⊂ 30% of training pool
- Same validation set across all experiments (fair comparison)

**Weighting:** 40-30-20-10 Hybrid Stratification
- 40% from Q4 (highest TADP - most complex)
- 30% from Q3
- 20% from Q2
- 10% from Q1 (zero TADP - negative examples)

---

**Author:** Luqman  
**Project:** AI6129 Pathogen Tracking System  
**Date:** January 2025  
**Version:** 2.0 (GEPA-optimised)

In [1]:
import re
import json
import random
import math
from pathlib import Path
from collections import Counter
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from google.colab import drive

In [2]:
# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)
print("Google Drive mounted")

Mounting Google Drive...
Mounted at /content/drive
Google Drive mounted


In [3]:
# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS FOR YOUR ENVIRONMENT
# =============================================================================

# Directory paths
GROUND_TRUTH_FOLDER = "/content/drive/MyDrive/AI6129/ground_truth"
OUTPUT_FOLDER = "/content/drive/MyDrive/AI6129/assay/validation_splits"

# Assay fields to count for TADP calculation
ASSAY_FIELDS = [
    "serotype",
    "mlst",
    "spi",
    "amr",
    "plasmid",
    "snp",
    "virulence_genes"
]

# Total expected samples
TOTAL_SAMPLES = 227

# Holdout ratio (held out for final evaluation after GEPA optimisation)
HOLDOUT_RATIO = 0.20  # 20% = ~45 samples

# =============================================================================
# GEPA-OPTIMISED SPLIT CONFIGURATION (DD-2025-003)
# =============================================================================

# Fixed validation set size (small, representative)
# GEPA guidance: "smallest valset that matches downstream task distribution"
VALIDATION_SIZE = 35  # ~8-9 samples per quartile  # changed

# Nested training split ratios (of training pool, NOT working set)  # changed
# Training pool = working set - validation set
NESTED_TRAINING_RATIOS = {
    "split_10": 0.12,  # ~18 samples from pool of ~148
    "split_20": 0.25,  # ~37 samples
    "split_30": 0.37,  # ~55 samples
}

# Hybrid stratification weights (favour complex examples)
# Q4 = highest TADP (most complex), Q1 = lowest/zero TADP (simplest/negative)
QUARTILE_WEIGHTS = {
    "Q4": 0.40,  # 40% from highest TADP quartile
    "Q3": 0.30,  # 30% from medium-high TADP
    "Q2": 0.20,  # 20% from medium-low TADP
    "Q1": 0.10,  # 10% from lowest/zero TADP (negative examples)
}

# Random seed for reproducibility
RANDOM_SEED = 42

## TADP Calculation Functions

These functions calculate the Total Assay Data Points (TADP) for each document.

In [4]:
def count_ast_entries(ast_data: List[dict]) -> int:
    """
    Count the number of antibiotic entries in AST data.

    Args:
        ast_data: List of AST dictionaries, each containing 'Antibiotics' list

    Returns:
        Total count of antibiotic entries
    """
    total = 0

    if not ast_data or not isinstance(ast_data, list):
        return 0

    for ast_entry in ast_data:
        if isinstance(ast_entry, dict):
            antibiotics = ast_entry.get("Antibiotics", [])
            if isinstance(antibiotics, list):
                total += len(antibiotics)

    return total


def count_assay_fields_in_isolate(isolate: dict) -> int:
    """
    Count assay data points in a single isolate entry.

    Args:
        isolate: Dictionary containing isolate data

    Returns:
        Count of assay data points
    """
    count = 0

    if not isinstance(isolate, dict):
        return 0

    # Count standard assay fields (list-based)
    for field in ASSAY_FIELDS:
        val = isolate.get(field, [])
        if isinstance(val, list):
            count += len(val)
        elif isinstance(val, str) and val and val.lower() != "null":
            count += 1

    # Count AST entries (nested structure)
    count += count_ast_entries(isolate.get("ast_data", []))

    return count


def calculate_tadp(ground_truth_data: dict) -> int:
    """
    Calculate Total Assay Data Points (TADP) from ground truth.

    Counts individual values from:
    - serotype, mlst, spi, amr, plasmid, snp, virulence_genes (list lengths)
    - ast_data (count of antibiotic entries)

    Sources:
    - isolates_with_linking (per-isolate assay data)
    - no_isolates_only_assayinformation (study-level assay data)

    Does NOT count isolate_without_linking (just IDs, no assay data).

    Args:
        ground_truth_data: The loaded JSON ground truth dictionary

    Returns:
        Total Assay Data Points count
    """
    total = 0

    # Count from isolates_with_linking
    isolates_with_linking = ground_truth_data.get("isolates_with_linking", [])
    if isinstance(isolates_with_linking, list):
        for isolate in isolates_with_linking:
            total += count_assay_fields_in_isolate(isolate)

    # Count from no_isolates_only_assayinformation
    nioai = ground_truth_data.get("no_isolates_only_assayinformation", {})
    if isinstance(nioai, dict) and nioai:
        total += count_assay_fields_in_isolate(nioai)

    return total


def categorise_document_sections(ground_truth_data: dict) -> str:
    """
    Categorise document by which sections contain data.

    Categories:
    - IWL: Only isolates_with_linking has data
    - IWOL: Only isolate_without_linking has data
    - NIOAI: Only no_isolates_only_assayinformation has data
    - Combined categories for multiple sections (e.g., IWL+NIOAI)
    - EMPTY: No sections have data

    Args:
        ground_truth_data: The loaded JSON ground truth dictionary

    Returns:
        Category string
    """
    has_iwl = False
    has_iwol = False
    has_nioai = False

    # Check isolates_with_linking
    iwl = ground_truth_data.get("isolates_with_linking", [])
    if isinstance(iwl, list) and len(iwl) > 0:
        has_iwl = True

    # Check isolate_without_linking
    iwol = ground_truth_data.get("isolate_without_linking", [])
    if isinstance(iwol, list) and len(iwol) > 0:
        has_iwol = True

    # Check no_isolates_only_assayinformation
    nioai = ground_truth_data.get("no_isolates_only_assayinformation", {})
    if isinstance(nioai, dict) and len(nioai) > 0:
        has_nioai = True

    # Build category string
    categories = []
    if has_iwl:
        categories.append("IWL")
    if has_iwol:
        categories.append("IWOL")
    if has_nioai:
        categories.append("NIOAI")

    if not categories:
        return "EMPTY"

    return "+".join(categories)

## Data Loading Functions

In [5]:
def load_ground_truth_files(folder_path: str) -> Dict[str, dict]:
    """
    Load all ground truth JSON files from a folder.

    Args:
        folder_path: Path to folder containing ground truth JSON files

    Returns:
        Dictionary mapping PMCID to ground truth data
    """
    ground_truth_dict = {}
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Ground truth folder not found: {folder_path}")

    json_files = list(folder.glob("*.json"))
    print(f"Found {len(json_files)} JSON files in {folder_path}")

    for json_file in json_files:
        # Extract PMCID from filename (assumes format: PMC{id}.json or PMC{id}_*.json)
        pmcid_match = re.match(r"(PMC\d+)", json_file.stem)
        if pmcid_match:
            pmcid = pmcid_match.group(1)
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                ground_truth_dict[pmcid] = data
            except json.JSONDecodeError as e:
                print(f"Warning: Could not parse {json_file.name}: {e}")
        else:
            print(f"Warning: Could not extract PMCID from {json_file.name}")

    return ground_truth_dict

## Stratification Functions

In [6]:
def assign_quartiles(ground_truth_dict: Dict[str, dict]) -> Dict[str, dict]:
    """
    Calculate TADP and assign quartiles to each document.

    Args:
        ground_truth_dict: Dictionary mapping PMCID to ground truth data

    Returns:
        Dictionary with TADP and quartile assignments added
    """
    # Calculate TADP for each document
    for pmcid, data in ground_truth_dict.items():
        data['tadp'] = calculate_tadp(data)
        data['category'] = categorise_document_sections(data)

    # Get sorted TADP values for quartile calculation
    tadp_values = sorted([data['tadp'] for data in ground_truth_dict.values()])
    n = len(tadp_values)

    # Calculate quartile boundaries
    q1_idx = n // 4
    q2_idx = n // 2
    q3_idx = (3 * n) // 4

    q1_threshold = tadp_values[q1_idx]
    q2_threshold = tadp_values[q2_idx]
    q3_threshold = tadp_values[q3_idx]

    # Assign quartiles
    for pmcid, data in ground_truth_dict.items():
        tadp = data['tadp']
        if tadp <= q1_threshold:
            data['quartile'] = 'Q1'
        elif tadp <= q2_threshold:
            data['quartile'] = 'Q2'
        elif tadp <= q3_threshold:
            data['quartile'] = 'Q3'
        else:
            data['quartile'] = 'Q4'

    return ground_truth_dict


def stratified_sample(
    pmcids: List[str],
    ground_truth_dict: Dict[str, dict],
    n_samples: int,
    weights: Dict[str, float],
    rng: random.Random
) -> List[str]:
    """
    Create a stratified sample based on TADP quartiles.

    Args:
        pmcids: List of PMCIDs to sample from
        ground_truth_dict: Dictionary with quartile assignments
        n_samples: Total number of samples to select
        weights: Dictionary of quartile weights (Q1, Q2, Q3, Q4)
        rng: Random number generator for reproducibility

    Returns:
        List of selected PMCIDs
    """
    # Group PMCIDs by quartile
    quartile_groups = {'Q1': [], 'Q2': [], 'Q3': [], 'Q4': []}
    for pmcid in pmcids:
        quartile = ground_truth_dict[pmcid]['quartile']
        quartile_groups[quartile].append(pmcid)

    # Shuffle each group
    for quartile in quartile_groups:
        rng.shuffle(quartile_groups[quartile])

    # Calculate target samples per quartile
    selected = []
    for quartile in ['Q4', 'Q3', 'Q2', 'Q1']:  # Priority order
        target = int(round(n_samples * weights[quartile]))
        available = quartile_groups[quartile]
        actual = min(target, len(available))
        selected.extend(available[:actual])

    # Handle rounding - add or remove to match exact target
    while len(selected) < n_samples:
        # Add from largest available quartile
        for quartile in ['Q4', 'Q3', 'Q2', 'Q1']:
            remaining = [p for p in quartile_groups[quartile] if p not in selected]
            if remaining:
                selected.append(remaining[0])
                break
        else:
            break  # No more samples available

    while len(selected) > n_samples:
        # Remove from smallest weight quartile
        for quartile in ['Q1', 'Q2', 'Q3', 'Q4']:
            in_quartile = [p for p in selected if ground_truth_dict[p]['quartile'] == quartile]
            if in_quartile:
                selected.remove(in_quartile[-1])
                break

    return selected

## GEPA-Optimised Split Creation  # changed

This is the key change: Fixed validation set with variable training sizes.

In [7]:
def create_gepa_optimised_splits(  # changed
    ground_truth_dict: Dict[str, dict],
    holdout_ratio: float,
    validation_size: int,
    training_ratios: Dict[str, float],
    weights: Dict[str, float],
    seed: int
) -> dict:
    """
    Create GEPA-optimised data splits with fixed validation and variable training.

    Key design principle (DD-2025-003):
    - GEPA needs LARGE training sets for diverse prompt exploration
    - GEPA needs SMALL validation sets for efficient Pareto score tracking
    - Same validation set across all experiments for fair comparison

    Args:
        ground_truth_dict: Dictionary with TADP and quartile assignments
        holdout_ratio: Fraction to hold out for final evaluation
        validation_size: Fixed number of validation samples
        training_ratios: Dict of split names to training pool ratios
        weights: Quartile sampling weights
        seed: Random seed for reproducibility

    Returns:
        Dictionary containing all split assignments and metadata
    """
    rng = random.Random(seed)
    all_pmcids = list(ground_truth_dict.keys())
    rng.shuffle(all_pmcids)

    n_total = len(all_pmcids)
    n_holdout = int(round(n_total * holdout_ratio))

    # Step 1: Create stratified holdout set
    holdout_set = stratified_sample(
        all_pmcids, ground_truth_dict, n_holdout, weights, rng
    )

    # Step 2: Working set = everything not in holdout
    working_set = [p for p in all_pmcids if p not in holdout_set]

    # Step 3: Create FIXED validation set from working set  # changed
    # This is the key change - validation is fixed, not the remainder
    validation_set = stratified_sample(
        working_set, ground_truth_dict, validation_size, weights, rng
    )

    # Step 4: Training pool = working set minus validation  # changed
    training_pool = [p for p in working_set if p not in validation_set]

    # Step 5: Create nested training splits from training pool  # changed
    # Largest split first, then subset for smaller splits
    n_pool = len(training_pool)

    # Calculate absolute sizes
    split_sizes = {
        name: int(round(n_pool * ratio))
        for name, ratio in training_ratios.items()
    }

    # Create largest split (30%) with stratification
    split_30 = stratified_sample(
        training_pool, ground_truth_dict, split_sizes["split_30"], weights, rng
    )

    # Create 20% as nested subset of 30%
    split_20 = stratified_sample(
        split_30, ground_truth_dict, split_sizes["split_20"], weights, rng
    )

    # Create 10% as nested subset of 20%
    split_10 = stratified_sample(
        split_20, ground_truth_dict, split_sizes["split_10"], weights, rng
    )

    return {
        "ground_truth_dict": ground_truth_dict,
        "holdout_set": holdout_set,
        "working_set": working_set,
        "validation_set": validation_set,  # changed: now fixed size
        "training_pool": training_pool,     # changed: new field
        "split_indices": {
            "split_10": split_10,
            "split_20": split_20,
            "split_30": split_30,
        },
        "metadata": {
            "total_documents": n_total,
            "holdout_size": len(holdout_set),
            "working_size": len(working_set),
            "validation_size": len(validation_set),  # changed
            "training_pool_size": len(training_pool),  # changed
            "split_10_size": len(split_10),
            "split_20_size": len(split_20),
            "split_30_size": len(split_30),
            "holdout_ratio": holdout_ratio,
            "quartile_weights": weights,
            "random_seed": seed,
            "design_decision": "DD-2025-003",  # changed
            "created_at": datetime.now().isoformat(),
        }
    }

## Reporting Functions

In [8]:
def print_split_summary(results: dict) -> None:
    """
    Print a comprehensive summary of the split assignments.
    """
    gt = results["ground_truth_dict"]
    meta = results["metadata"]

    print("\n" + "=" * 70)
    print("GEPA-OPTIMISED SPLIT SUMMARY (DD-2025-003)")  # changed
    print("=" * 70)

    print(f"\nTotal documents: {meta['total_documents']}")
    print(f"Random seed: {meta['random_seed']}")

    # Holdout summary
    print(f"\n{'-' * 70}")
    print("HOLDOUT SET (Final Evaluation)")
    print(f"{'-' * 70}")
    holdout_tadps = [gt[p]['tadp'] for p in results['holdout_set']]
    holdout_quartiles = Counter([gt[p]['quartile'] for p in results['holdout_set']])
    print(f"  Size: {meta['holdout_size']} documents ({meta['holdout_ratio']*100:.0f}%)")
    print(f"  TADP range: {min(holdout_tadps)} - {max(holdout_tadps)}")
    print(f"  Mean TADP: {sum(holdout_tadps)/len(holdout_tadps):.1f}")
    print(f"  Quartile distribution: {dict(holdout_quartiles)}")

    # Working set summary
    print(f"\n{'-' * 70}")
    print("WORKING SET")
    print(f"{'-' * 70}")
    print(f"  Size: {meta['working_size']} documents")

    # Validation set summary (FIXED)  # changed
    print(f"\n{'-' * 70}")
    print("VALIDATION SET (Fixed for all experiments)")  # changed
    print(f"{'-' * 70}")
    val_tadps = [gt[p]['tadp'] for p in results['validation_set']]
    val_quartiles = Counter([gt[p]['quartile'] for p in results['validation_set']])
    print(f"  Size: {meta['validation_size']} documents")  # changed
    print(f"  TADP range: {min(val_tadps)} - {max(val_tadps)}")
    print(f"  Mean TADP: {sum(val_tadps)/len(val_tadps):.1f}")
    print(f"  Quartile distribution: {dict(val_quartiles)}")

    # Training pool summary  # changed
    print(f"\n{'-' * 70}")
    print("TRAINING POOL (Available for nested splits)")  # changed
    print(f"{'-' * 70}")
    pool_tadps = [gt[p]['tadp'] for p in results['training_pool']]
    pool_quartiles = Counter([gt[p]['quartile'] for p in results['training_pool']])
    print(f"  Size: {meta['training_pool_size']} documents")  # changed
    print(f"  TADP range: {min(pool_tadps)} - {max(pool_tadps)}")
    print(f"  Mean TADP: {sum(pool_tadps)/len(pool_tadps):.1f}")
    print(f"  Quartile distribution: {dict(pool_quartiles)}")

    # Nested training configurations  # changed
    print(f"\n{'-' * 70}")
    print("GEPA EXPERIMENT CONFIGURATIONS")  # changed
    print(f"{'-' * 70}")

    for split_name in ["split_10", "split_20", "split_30"]:
        split_pmcids = results["split_indices"][split_name]
        split_tadps = [gt[p]['tadp'] for p in split_pmcids]
        split_quartiles = Counter([gt[p]['quartile'] for p in split_pmcids])

        pct = split_name.replace("split_", "")
        train_size = len(split_pmcids)
        val_size = meta['validation_size']  # changed: fixed validation
        ratio = train_size / val_size if val_size > 0 else 0  # changed

        print(f"\n{pct}% Configuration:")
        print(f"  Training:    {train_size:3d} samples")
        print(f"  Validation:  {val_size:3d} samples (fixed)")  # changed
        print(f"  Train:Val ratio: {ratio:.2f}:1")  # changed
        print(f"  Training TADP range: {min(split_tadps)} - {max(split_tadps)}")
        print(f"  Training mean TADP: {sum(split_tadps)/len(split_tadps):.1f}")
        q_str = ", ".join([f"{q}={split_quartiles.get(q, 0)}" for q in ['Q4', 'Q3', 'Q2', 'Q1']])
        print(f"  Training quartile dist: {q_str}")

## Save Functions

In [9]:
def save_split_assignments(results: dict, output_folder: str) -> None:
    """
    Save split assignments to JSON and CSV files.
    """
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    # Prepare data for JSON export
    export_data = {
        "metadata": results["metadata"],
        "holdout_set": results["holdout_set"],
        "validation_set": results["validation_set"],  # changed
        "training_pool": results["training_pool"],    # changed
        "split_indices": results["split_indices"],
        "document_details": {
            pmcid: {
                "tadp": data["tadp"],
                "quartile": data["quartile"],
                "category": data["category"]
            }
            for pmcid, data in results["ground_truth_dict"].items()
        }
    }

    # Save JSON  # changed filename
    json_path = output_path / "assay_tadp_gepa_optimised_splits.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2)
    print(f"\nSplit assignments saved to: {json_path}")

    # Create CSV for easy viewing  # changed filename
    csv_path = output_path / "assay_tadp_gepa_optimised_splits.csv"
    with open(csv_path, 'w', encoding='utf-8') as f:
        f.write("pmcid,tadp,quartile,category,holdout,validation,training_pool,split_10,split_20,split_30\n")  # changed
        for pmcid, data in results["ground_truth_dict"].items():
            in_holdout = 1 if pmcid in results["holdout_set"] else 0
            in_validation = 1 if pmcid in results["validation_set"] else 0  # changed
            in_pool = 1 if pmcid in results["training_pool"] else 0  # changed
            in_10 = 1 if pmcid in results["split_indices"]["split_10"] else 0
            in_20 = 1 if pmcid in results["split_indices"]["split_20"] else 0
            in_30 = 1 if pmcid in results["split_indices"]["split_30"] else 0
            f.write(f"{pmcid},{data['tadp']},{data['quartile']},{data['category']},")
            f.write(f"{in_holdout},{in_validation},{in_pool},{in_10},{in_20},{in_30}\n")  # changed
    print(f"CSV export saved to: {csv_path}")

## Main Execution

In [10]:
def main() -> dict:
    """
    Main function to create GEPA-optimised stratified splits.
    """
    print("=" * 70)
    print("ASSAY GROUND TRUTH: GEPA-OPTIMISED STRATIFICATION")  # changed
    print("Design Decision: DD-2025-003")  # changed
    print("=" * 70)

    # Step 1: Load ground truth files
    print(f"\n{'-' * 70}")
    print("STEP 1: Loading Ground Truth Files")
    print(f"{'-' * 70}")
    ground_truth_dict = load_ground_truth_files(GROUND_TRUTH_FOLDER)
    print(f"Loaded {len(ground_truth_dict)} ground truth documents")

    if len(ground_truth_dict) != TOTAL_SAMPLES:
        print(f"WARNING: Expected {TOTAL_SAMPLES} samples, found {len(ground_truth_dict)}")

    # Step 2: Calculate TADP and assign quartiles
    print(f"\n{'-' * 70}")
    print("STEP 2: Calculating TADP and Assigning Quartiles")
    print(f"{'-' * 70}")
    ground_truth_dict = assign_quartiles(ground_truth_dict)

    # Print quartile summary
    quartile_stats = {}
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        q_docs = [p for p, d in ground_truth_dict.items() if d['quartile'] == q]
        q_tadps = [ground_truth_dict[p]['tadp'] for p in q_docs]
        quartile_stats[q] = {
            'count': len(q_docs),
            'min': min(q_tadps) if q_tadps else 0,
            'max': max(q_tadps) if q_tadps else 0,
            'mean': sum(q_tadps) / len(q_tadps) if q_tadps else 0
        }

    print("\nQuartile   Count    Min TADP   Max TADP   Mean TADP    Weight")
    print("-" * 70)
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        s = quartile_stats[q]
        w = int(QUARTILE_WEIGHTS[q] * 100)
        print(f"{q:10s} {s['count']:<8d} {s['min']:<10d} {s['max']:<10d} {s['mean']:<12.1f} {w}%")

    # Step 3: Create GEPA-optimised splits  # changed
    print(f"\n{'-' * 70}")
    print("STEP 3: Creating GEPA-Optimised Splits")  # changed
    print(f"{'-' * 70}")
    print(f"\nKey change (DD-2025-003): Fixed validation ({VALIDATION_SIZE} samples)")  # changed
    print("GEPA requires small validation for efficient Pareto tracking")

    results = create_gepa_optimised_splits(  # changed
        ground_truth_dict=ground_truth_dict,
        holdout_ratio=HOLDOUT_RATIO,
        validation_size=VALIDATION_SIZE,  # changed
        training_ratios=NESTED_TRAINING_RATIOS,
        weights=QUARTILE_WEIGHTS,
        seed=RANDOM_SEED
    )

    # Step 4: Verify nested structure
    print(f"\n{'-' * 70}")
    print("STEP 4: Verifying Nested Structure")
    print(f"{'-' * 70}")
    split_10 = set(results["split_indices"]["split_10"])
    split_20 = set(results["split_indices"]["split_20"])
    split_30 = set(results["split_indices"]["split_30"])

    nested_ok = split_10.issubset(split_20) and split_20.issubset(split_30)
    print(f"10% ⊂ 20%: {split_10.issubset(split_20)}")
    print(f"20% ⊂ 30%: {split_20.issubset(split_30)}")
    print(f"Nested structure valid: {nested_ok}")

    # Step 5: Print summary
    print_split_summary(results)

    # Step 6: Save results
    print(f"\n{'-' * 70}")
    print("STEP 5: Saving Split Assignments")
    print(f"{'-' * 70}")
    save_split_assignments(results, OUTPUT_FOLDER)

    print("\n" + "=" * 70)
    print("SPLIT CREATION COMPLETE")
    print("=" * 70)

    return results

In [11]:
if __name__ == "__main__":
    results = main()

ASSAY GROUND TRUTH: GEPA-OPTIMISED STRATIFICATION
Design Decision: DD-2025-003

----------------------------------------------------------------------
STEP 1: Loading Ground Truth Files
----------------------------------------------------------------------
Found 227 JSON files in /content/drive/MyDrive/AI6129/ground_truth
Loaded 227 ground truth documents

----------------------------------------------------------------------
STEP 2: Calculating TADP and Assigning Quartiles
----------------------------------------------------------------------

Quartile   Count    Min TADP   Max TADP   Mean TADP    Weight
----------------------------------------------------------------------
Q1         92       0          0          0.0          10%
Q2         22       1          3          2.2          20%
Q3         57       4          24         11.8         30%
Q4         56       25         292        71.7         40%

----------------------------------------------------------------------
STEP 3: 

## Verification: Check GEPA Ratios  # changed

In [12]:
# Verify GEPA-friendly ratios  # changed
if results:
    print("GEPA Ratio Verification:")
    print("=" * 50)
    val_size = len(results["validation_set"])

    for split_name in ["split_10", "split_20", "split_30"]:
        train_size = len(results["split_indices"][split_name])
        ratio = train_size / val_size
        pct = split_name.replace("split_", "")

        # GEPA prefers train >= val
        status = "OK" if ratio >= 0.5 else "WARNING: train < val"
        print(f"{pct}%: Train={train_size}, Val={val_size}, Ratio={ratio:.2f}:1 [{status}]")

    print("\nNote: GEPA works best when training set >= validation set")
    print("The fixed validation allows fair comparison across experiments")

GEPA Ratio Verification:
10%: Train=18, Val=35, Ratio=0.51:1 [OK]
20%: Train=37, Val=35, Ratio=1.06:1 [OK]
30%: Train=54, Val=35, Ratio=1.54:1 [OK]

Note: GEPA works best when training set >= validation set
The fixed validation allows fair comparison across experiments


## Optional: View Sample Documents by Split

In [13]:
# Uncomment to view sample documents
if results:
    print("\nSample documents in 30% training split:")
    for pmcid in results["split_indices"]["split_30"][:5]:
        doc = results["ground_truth_dict"][pmcid]
        print(f"  {pmcid}: TADP={doc['tadp']}, Quartile={doc['quartile']}, Category={doc['category']}")

    print("\nSample documents in validation set:")
    for pmcid in results["validation_set"][:5]:
        doc = results["ground_truth_dict"][pmcid]
        print(f"  {pmcid}: TADP={doc['tadp']}, Quartile={doc['quartile']}, Category={doc['category']}")


Sample documents in 30% training split:
  PMC9137667: TADP=28, Quartile=Q4, Category=IWL
  PMC9409446: TADP=25, Quartile=Q4, Category=IWL
  PMC9220226: TADP=79, Quartile=Q4, Category=IWL
  PMC9769515: TADP=90, Quartile=Q4, Category=IWL+IWOL
  PMC7603275: TADP=27, Quartile=Q4, Category=IWL

Sample documents in validation set:
  PMC6035434: TADP=58, Quartile=Q4, Category=IWL
  PMC9216381: TADP=60, Quartile=Q4, Category=IWL
  PMC9513250: TADP=123, Quartile=Q4, Category=IWL+IWOL
  PMC8311177: TADP=71, Quartile=Q4, Category=IWL
  PMC9708683: TADP=124, Quartile=Q4, Category=IWL+IWOL
